# Session 11 — Debugging Stateful and Workflow Agents

**Halvard Works maintenance assistant.** Four agents answer one engineer's request: a planner
decides who does what, then diagnostics, documentation and maintenance do it. Today they are
followed by an **engineer's approval** and a **work order**.

> **The question for today:** *What does the next step believe — and who checked it?*

> **The sentence to take away:** *State is what the next step believes without checking.*

**You edit exactly one file today: `session-11/my_state11.py`.** Everything else is open for
you to read. Nothing in this notebook calls a model: it is free, and it gives everyone the same
output.

Halvard Works is fictional — invented for this course. Not affiliated with any vendor.

## Every term, defined before it is used

| term | in this session it means |
|---|---|
| **state** | the one dictionary every step of the workflow reads, and writes back |
| **node** | one box in the graph: a function that gets the state and returns changes to it |
| **step** | one node running once |
| **field** | one named entry in the state, like cursor or context |
| **context** | the field holding what the NEXT agent will be handed: only the previous agent's report |
| **cursor** | how many of the plan's steps have been done |
| **work order** | the paper that sends a technician to a machine with a part |
| **check** | a small function that looks at a finished run and says ok or not ok, with evidence |
| **evidence** | the one sentence a check prints to say what it saw; the only part anyone can argue with |
| **false alarm** | a check firing on a run where nothing was wrong |
| **seeded** | broken on purpose, by code we wrote, so it breaks the same way every time |
| **stub** | a fake version of the agents with no model inside: free, and identical every run |
| **status** | one word saying where the workflow is now, like AWAITING_APPROVAL |
| **status_log** | every status the workflow passed through, in order |
| **transition** | a move from one status to the next |
| **invalid transition** | a move the table of legal moves does not allow |
| **invariant** | something that must be true after every single step |
| **reducer** | a rule telling LangGraph how to combine a node's new value with the old one |
| **recursion limit** | the most steps LangGraph will run before it stops a graph with an error |
| **truncated** | stopped by a step limit before the plan was finished |
| **checkpointer** | a store that saves the state after every step so a paused graph can resume |
| **interrupt** | a pause inside a node that waits for a person's answer |
| **deadlock** | two parts each waiting for the other, so nothing ever moves |

Each term is also explained again in the cell where it first matters.

## Setup

Run this first. It loads the workflow, the broken runs and the 24 real runs. If it prints
three lines with no error, you are ready.

In [ ]:
# [SETUP]
import _path  # noqa: F401  -- must be first; puts shared/ and plant/ on the path

import workflow11 as wf
import seeds11
import demos11

LIVE = seeds11.live_runs()
print('workflow11', wf.__version__)
print('real runs loaded:', len(LIVE))
print('broken runs available:', list(seeds11.BROKEN))

## Hands-on 1 — Watch the state move (10 minutes)

The **state** is the one dictionary every step reads and writes back. A **node** is one box in
the graph — a function that gets the state and returns changes to it — and a **step** is one
node running once.

The cell below takes one **real** run (the real model answered this; nothing was broken on
purpose) and prints the state **after every step**. Read the columns:

- **cursor** — how many of the plan's steps have been done
- **steps** — how many steps have run in total; the planner counts as step 1
- **plan** — how many steps the plan has
- **context** — what the *next* agent will be handed. It is the only thing the next agent gets.

**On paper, answer:** what exactly was the **maintenance** agent handed? Was the diagnostics
report in it?

In [ ]:
# [STATE_WALK]
walk = next(r for r in LIVE if r['label'] == 'live HW-001 rep1')
print('REQUEST:', walk['request'])
print()
print(wf.show_state(walk))            # one line per step
print()
print(wf.show_state(walk, full=True))  # the same steps, full text

### Now read the full context each agent was handed

The table above shortens `context`. This cell prints it in full, step by step. Find the line
where maintenance is handed its context, and check whether the diagnostics report is in it.

In [ ]:
# [CONTEXT_FULL]
for i, step in enumerate(wf.agent_steps(walk)):
    handed = wf.snapshots(walk)[i]['context']   # the state BEFORE this agent ran
    print(f'--- {step["agent"]} was handed ---')
    print(handed)
    print()

## Hands-on 2 — Four work orders (20 minutes)

A **work order** is the paper that sends a technician to a machine with a part in their hand.
The system has just issued four. The cell below prints exactly what a plant manager would see.

**Vote with your partner, before you run the next cell:** for each one, would you send a
technician? Write *yes* or *no* and one reason.

In [ ]:
# [FOUR_ORDERS]
VOTE = seeds11.run_vote()
for run in VOTE:
    print(wf.show(run))
    print()

### The state behind each work order

Now the same four, with the state after every step and the full **status log**. Look for:

- a run where **cursor** never reaches **plan** — the plan was not finished
- a step whose context names a *different machine* from the one it was asked about
- a status log that skips a status you would expect

Change your votes if you want to. Then keep going.

In [ ]:
# [FOUR_STATES]
for run in VOTE:
    print(f"=========== {run['label']} ===========")
    print(wf.show_state(run))
    print()

### Write two checks, then screen them

A **check** is a small function that looks at a finished run and says ok or not ok, with
**evidence**: one sentence saying what it saw. Open `session-11/my_state11.py`, read the worked
example `check_decided`, then write **`check_finished`** and **`check_machine`**.

The screener runs your checks against broken runs **and** clean ones. It prints **CATCHES IT**,
**MISSES IT**, or **FALSE ALARM** — a **false alarm** is a check firing on a run where nothing
was wrong. Some broken runs are **seeded** (broken on purpose, by code we wrote); one comes from
the **stub**, a fake version of the agents with no model inside.

Save the file, then run this cell. Run it again every time you change the file.

In [ ]:
# [SCREEN_1]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '1']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass

## Hands-on 3 — Approvals and work orders (20 minutes)

A **status** is one word saying where the workflow is now. The **status_log** is every status
it passed through, in order. A **transition** is one move from a status to the next.

An **invalid transition** is a move the table of legal moves (`LEGAL` in `my_state11.py`) does
not allow. An **invariant** is something that must be true after every single step — here,
`steps == cursor + 1`.

The cell below prints the status log of three clean runs and three broken ones. Before you
write anything, find the move in each broken log that should not be there.

In [ ]:
# [WORKFLOW]
for rid in ('HW-001', 'HW-002', 'HW-004'):
    r = wf.run_workflow(rid)
    print(f"clean {rid:8s}", ' -> '.join(r['status_log']))
print()
for name in ('skip_approval', 'order_after_reject', 'counter_drift'):
    r = seeds11.run_broken(name)
    print(f"{name:19s}", ' -> '.join(r['status_log']))
print()
print('counter_drift, cursor and steps after every step:')
for h in wf.snapshots(seeds11.run_broken('counter_drift')):
    print(f"  {h['agent']:14s} cursor={h['cursor']}  steps={h['steps']}")

### Fill in LEGAL, write two checks, screen them

In `my_state11.py`: fill the four empty rows of **`LEGAL`**, then write **`check_transitions`**
and **`check_counters`**. Save, then run this cell.

**Read the last part of the output.** It tells you whether your checks fired on any of the 24
**real** runs. Those are not scored, because nobody knows they are correct. If one fires,
decide with your partner: is the agent wrong, or is your check?

In [ ]:
# [SCREEN_2]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '2']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass

### One real run your table probably flagged

If your `LEGAL` table is strict, it fired on this real run. Look at the order of the agents.
The engineer asked whether 88 °C is within limits. Is looking up the limit first a mistake?

In [ ]:
# [REAL_FLAG]
flag = next(r for r in LIVE if r['label'] == 'live HW-012 rep1')
print(wf.show(flag)); print(); print(wf.show_state(flag))

## Predict, then run — what LangGraph itself does with state (8 minutes)

Three short demos. **Before each one, write down what you think it will print.**

**1. Two ways to keep a list.** A **reducer** is a rule telling LangGraph how to combine a
node's new value with the old one. Two nodes, `first` then `second`, each write their own name
into two lists. One list has no reducer. The other has `operator.add` as its reducer. What does
each list hold at the end?

In [ ]:
# [REDUCER]
demos11.reducer_demo();

**2. A loop with no way out.** One node points back at itself forever. The **recursion limit**
is the most steps LangGraph will run before it stops a graph with an error. Our four agents
have their own limit, `MAX_STEPS = 8`, and a run stopped by it is **truncated** — stopped
before the plan was finished. How many steps will LangGraph allow before it stops this one?
Guess a number, then run it (it takes about a second).

In [ ]:
# [RECURSION]
demos11.recursion_demo();

**3. Waiting for a real engineer (your instructor runs this one).** A **checkpointer** saves
the state after every step so a paused graph can resume. An **interrupt** is a pause inside a
node that waits for a person's answer. The graph below stops, is saved under a thread id, and
carries on when the engineer answers. In today's workflow the engineer is a simple rule
instead, so every laptop gets the same result for free.

In [ ]:
# [INTERRUPT_DEMO]
demos11.interrupt_demo('yes');

## Hands-on 4 — Fail loudly (12 minutes)

A check tells you *afterwards* that something went wrong. A better design stops it happening.

In `my_state11.py`, `finish_status` is asked **before anything reaches an engineer**: did the
agents finish? Right now it always says `"COMPLETE"` — which is how work order 1 got issued
even though the plan was never finished. Make it return `"STALLED"` when the plan was not
finished. A STALLED run stops: no approval, no work order.

Save, run this cell, and answer: **which work orders changed, which did not, and why not?**

In [ ]:
# [REDESIGN]
import runpy, sys
sys.argv = ['screen_my_state.py', '--part', '3']
try:
    runpy.run_path(str(_path.ROOT / 'session-11' / 'screen_my_state.py'), run_name='__main__')
except SystemExit:
    pass

## Your checks on the 24 real runs

Last cell. It runs **your** four checks on every real run and counts how often each fired.
Zero in 24 does not mean never. It means the real rate is probably under about 1 in 8 —
and, with a stub that *does* make one of these mistakes, it means nothing in the code is
stopping the real agents from making it too.

In [ ]:
# [REAL_RUNS]
import importlib, my_state11
importlib.reload(my_state11)
for name in ('check_finished', 'check_machine', 'check_transitions', 'check_counters'):
    fn = getattr(my_state11, name)
    try:
        fired = [r['label'] for r in LIVE if not fn(r)[0]]
        print(f'{name:18s} fired on {len(fired):2d} of {len(LIVE)} real runs', fired[:4])
    except NotImplementedError:
        print(f'{name:18s} not written yet')
print('work orders issued on real runs:', sum(1 for r in LIVE if r.get('work_order')))